# Imports

In [24]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying input length

## load data

In [27]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/vary_input'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'auxi_type', 'auxi_mode', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'seq_len', 'pred_len'], inplace=True)
df.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,auxi_type,auxi_mode,lradj,patience,train_epochs,mse,mae,exp_dir
1056,FreTS,192,96,Weather,0.0005,1.0,0.0,all,0,0,1.0,MAE,32,complex,fft,type1,3,10,0.158995,0.213995,/data/home/Licheng/workspace/TSF-PCA/results_P...
387,FreTS,192,192,Weather,0.0005,1.0,0.0,all,0,0,1.0,MAE,32,complex,fft,type1,3,10,0.201555,0.256462,/data/home/Licheng/workspace/TSF-PCA/results_P...
777,FreTS,192,336,Weather,0.0005,1.0,0.0,all,0,0,1.0,MAE,32,complex,fft,type1,3,10,0.250179,0.297790,/data/home/Licheng/workspace/TSF-PCA/results_P...
382,FreTS,192,720,Weather,0.0005,1.0,0.0,all,0,0,1.0,MAE,32,complex,fft,type1,3,10,0.320959,0.350454,/data/home/Licheng/workspace/TSF-PCA/results_P...


## preprocess

In [28]:
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_best = pd.read_csv(f'{save_root}/best_finetune_full_each.csv')

base = baselines.copy()
base = base[
    ((base.data_id == 'Weather') & (base.model == 'iTransformer'))
]
base['seq_len'] = 96
base['label'] = 'DF'

best = finetunes_best.copy()
best = best[
    ((best.data_id == 'Weather_PCA') & (best.model == 'iTransformer'))
]
best['seq_len'] = 96
best['label'] = 'PDF'
best = best[best.pred_len != 'Avg']
best['data_id'] = best['data_id'].str.replace('_PCA', '')


df2 = df.copy()
df2 = df2[
    ((df2.seq_len != 96) & (df2.model == 'iTransformer') & (df2.data_id.isin(['Weather', 'Weather_PCA']))) |
    ((df2.model == 'PatchTST') & (df2.data_id.isin(['Weather', 'Weather_PCA'])))
]

min_mse_idx = df2.groupby(['model', 'data_id', 'seq_len', 'pred_len'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]
df2['label'] = df2['data_id'].apply(lambda x: 'PDF' if "_PCA" in x else 'DF')
df2['data_id'] = df2['data_id'].str.replace('_PCA', '')

columns = ['model', 'data_id', 'seq_len', 'pred_len', 'label', 'mse', 'mae']
vary_seq = pd.concat([base[columns], best[columns], df2[columns]], ignore_index=True)
vary_seq['seq_len'] = vary_seq['seq_len'].astype(int)
vary_seq['pred_len'] = vary_seq['pred_len'].astype(int)

dst_order = ['Weather']
vary_seq['data_id'] = pd.Categorical(vary_seq['data_id'], categories=dst_order, ordered=True)

model_order = ['iTransformer', 'PatchTST']
vary_seq['model'] = pd.Categorical(vary_seq['model'], categories=model_order, ordered=True)


vary_seq_avg = vary_seq.groupby(['model', 'data_id', 'seq_len', 'label']).mean(numeric_only=True).reset_index()
vary_seq_avg['pred_len'] = 'Avg'
vary_seq = pd.concat([vary_seq, vary_seq_avg], ignore_index=True)

# sl_order = ['96', '192', '336', '720']
# vary_seq['seq_len'] = pd.Categorical(vary_seq['seq_len'], categories=sl_order, ordered=True)

# pl_order = ['96', '192', '336', '720', 'Avg']
# vary_seq['pred_len'] = pd.Categorical(vary_seq['pred_len'], categories=pl_order, ordered=True)

vary_seq.sort_values(by=['model', 'data_id', 'seq_len', 'label', 'pred_len'], inplace=True)

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
vary_seq.round(3).to_csv(f'{save_root}/vary_seq.csv', index=False, float_format='%.3f')
vary_seq

/tmp/ipykernel_751009/1151328204.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  vary_seq_avg = vary_seq.groupby(['model', 'data_id', 'seq_len', 'label']).mean(numeric_only=True).reset_index()


,model,data_id,seq_len,pred_len,label,mse,mae
0,iTransformer,Weather,96,96,DF,0.171432,0.210468
1,iTransformer,Weather,96,192,DF,0.246395,0.278306
2,iTransformer,Weather,96,336,DF,0.296230,0.312968
3,iTransformer,Weather,96,720,DF,0.362297,0.352779
64,iTransformer,Weather,96,Avg,DF,0.269089,0.288630
4,iTransformer,Weather,96,96,PDF,0.162750,0.201556
5,iTransformer,Weather,96,192,PDF,0.213842,0.247811
6,iTransformer,Weather,96,336,PDF,0.273705,0.293638
7,iTransformer,Weather,96,720,PDF,0.351255,0.344143
65,iTransformer,Weather,96,Avg,PDF,0.250388,0.271787
